# Word-level sign language training (hands + posture)

This Colab-ready notebook trains a video classifier for words such as **Good**, **Morning**, and **Night**. It uses the current MediaPipe Tasks Hand Landmarker and Pose Landmarker APIs, then learns motion across time with a browser-portable temporal CNN. In Google Colab it automatically mounts Drive and reads `/content/drive/MyDrive/sign_language_model/datasets.zip`.

Expected layout inside `datasets.zip` (extra nested folders are allowed):
```text
datasets/videos/
  Good/**.mp4
  Morning/**.mp4
  Night/**.mp4
```
The first folder below the discovered `videos` directory is treated as the class label. Run the cells from top to bottom. Trained Python and web artifacts are saved back to `MyDrive/sign_language_model`.

In [ ]:
# Colab-compatible pins. Run once, then restart the session.
%pip install -q "numpy==2.0.2" "pandas==2.2.3" "protobuf==5.29.5" "tensorflow==2.20.0" "mediapipe==1.0.1" "opencv-contrib-python==4.11.0.86"

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import shutil
import sys
import urllib.request
import warnings
import zipfile

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore", category=UserWarning)

SEQ_LEN = 40                 # frames supplied to the model
MAX_SAMPLE_FRAMES = 80      # MediaPipe inferences per source video
RANDOM_SEED = 42
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
CACHE_VERSION = "tasks_pose33_hands21_shoulder_norm_v2"

IN_COLAB = "google.colab" in sys.modules

def find_video_dataset(search_root: Path) -> Path:
    candidates = []
    for directory in search_root.rglob("videos"):
        if directory.is_dir():
            count = sum(
                1 for path in directory.rglob("*")
                if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
            )
            if count:
                candidates.append((count, directory))
    if not candidates:
        raise FileNotFoundError(
            f"No folder named 'videos' containing supported video files was found below {search_root}"
        )
    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[0][1]

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/sign_language_model")
    DATASET_ZIP = DRIVE_DIR / "datasets.zip"
    EXTRACT_ROOT = Path("/content/sign_language_dataset")
    SIGNATURE_FILE = EXTRACT_ROOT / ".datasets_zip_signature"

    if not DATASET_ZIP.exists():
        raise FileNotFoundError(f"Upload the dataset ZIP here: {DATASET_ZIP}")

    zip_stat = DATASET_ZIP.stat()
    zip_signature = f"{zip_stat.st_size}:{zip_stat.st_mtime_ns}"
    saved_signature = SIGNATURE_FILE.read_text().strip() if SIGNATURE_FILE.exists() else None
    if saved_signature != zip_signature:
        if EXTRACT_ROOT.exists():
            shutil.rmtree(EXTRACT_ROOT)
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {DATASET_ZIP} to Colab local storage...")
        with zipfile.ZipFile(DATASET_ZIP, "r") as archive:
            archive.extractall(EXTRACT_ROOT)
        SIGNATURE_FILE.write_text(zip_signature)

    PROJECT_DIR = DRIVE_DIR
    DATASET_DIR = find_video_dataset(EXTRACT_ROOT)
    CACHE_DIR = DRIVE_DIR / "landmark_cache"
    OUTPUT_DIR = DRIVE_DIR / "models" / "word_sign"
    WEB_EXPORT_DIR = DRIVE_DIR / "web_export" / "models" / "word-sign"
else:
    PROJECT_DIR = Path(r"C:\Users\kenne\Desktop\em-res\FSL-OD\handspeak")
    DATASET_DIR = PROJECT_DIR / "datasets" / "videos"
    CACHE_DIR = PROJECT_DIR / "datasets" / "landmark_cache"
    OUTPUT_DIR = PROJECT_DIR / "models" / "word_sign"
    WEB_EXPORT_DIR = PROJECT_DIR / "public" / "models" / "word-sign"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEB_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
MEDIAPIPE_MODEL_DIR = CACHE_DIR / "mediapipe_tasks"
MEDIAPIPE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
HAND_LANDMARKER_PATH = MEDIAPIPE_MODEL_DIR / "hand_landmarker.task"
POSE_LANDMARKER_PATH = MEDIAPIPE_MODEL_DIR / "pose_landmarker_full.task"

def download_if_missing(url: str, destination: Path):
    if not destination.exists():
        print(f"Downloading {destination.name}...")
        urllib.request.urlretrieve(url, destination)

download_if_missing(
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
    HAND_LANDMARKER_PATH,
)
download_if_missing(
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
    POSE_LANDMARKER_PATH,
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print("TensorFlow:", tf.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Running in Colab:", IN_COLAB)
print("Dataset:", DATASET_DIR)
print("Python model output:", OUTPUT_DIR)
print("Web model output:", WEB_EXPORT_DIR)

## 1. Discover and inspect the videos

Labels are inferred automatically. A minimum of roughly 20 varied videos per word is recommended; more signers, backgrounds, distances, and lighting conditions generally improve real-world accuracy.

In [ ]:
def discover_videos(dataset_dir: Path) -> pd.DataFrame:
    if not dataset_dir.exists():
        raise FileNotFoundError(f"Dataset folder not found: {dataset_dir}")

    rows = []
    for path in sorted(dataset_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS:
            relative = path.relative_to(dataset_dir)
            if len(relative.parts) < 2:
                print(f"Skipping video without a label folder: {path.name}")
                continue
            rows.append({"path": path, "label": relative.parts[0]})

    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError(f"No supported videos found below {dataset_dir}")
    return frame

videos_df = discover_videos(DATASET_DIR)
labels = sorted(videos_df["label"].unique().tolist())
label_to_index = {label: index for index, label in enumerate(labels)}
index_to_label = {index: label for label, index in label_to_index.items()}
videos_df["label_index"] = videos_df["label"].map(label_to_index)

counts = videos_df["label"].value_counts().sort_index()
display(counts.rename("videos").to_frame())
ax = counts.plot.bar(title=f"Dataset: {len(videos_df)} videos, {len(labels)} words", color="#4472C4")
ax.set_xlabel("Word")
ax.set_ylabel("Number of videos")
plt.xticks(rotation=0)
plt.show()

if len(labels) < 2:
    raise ValueError("Classification requires at least two word folders/classes.")
if counts.min() < 7:
    print("WARNING: At least one class has fewer than 7 videos; stratified train/validation/test splitting may fail.")

## 2. Extract normalized hand landmarks and posture

Each frame contains 21×3 coordinates for each hand, 33×4 pose values (x/y/z/visibility), and three presence flags. Coordinates are centered between the shoulders and scaled by shoulder width. Missing landmarks are stored as zeros and explicitly marked by the flags. Results are cached so later runs are much faster.

In [ ]:
HAND_VALUES = 21 * 3
POSE_VALUES = 33 * 4
FEATURE_DIM = HAND_VALUES * 2 + POSE_VALUES + 3

vision = mp.tasks.vision
BaseOptions = mp.tasks.BaseOptions

def create_hand_landmarker(running_mode):
    options = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(HAND_LANDMARKER_PATH)),
        running_mode=running_mode,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return vision.HandLandmarker.create_from_options(options)

def create_pose_landmarker(running_mode):
    options = vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(POSE_LANDMARKER_PATH)),
        running_mode=running_mode,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )
    return vision.PoseLandmarker.create_from_options(options)

class VideoLandmarkers:
    def __init__(self):
        self.hand = create_hand_landmarker(vision.RunningMode.VIDEO)
        self.pose = create_pose_landmarker(vision.RunningMode.VIDEO)
        self.timestamp_ms = 0

    def process(self, rgb):
        self.timestamp_ms += 33
        image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
        hand_result = self.hand.detect_for_video(image, self.timestamp_ms)
        pose_result = self.pose.detect_for_video(image, self.timestamp_ms)
        return hand_result, pose_result

    def close(self):
        self.hand.close()
        self.pose.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

def _hand_array(hand_landmarks):
    if hand_landmarks is None:
        return np.zeros((21, 3), dtype=np.float32)
    return np.asarray([[p.x, p.y, p.z] for p in hand_landmarks], dtype=np.float32)

def _pose_array(pose_landmarks):
    if pose_landmarks is None:
        return np.zeros((33, 4), dtype=np.float32)
    return np.asarray([
        [p.x, p.y, p.z, float(getattr(p, "visibility", 0.0) or 0.0)]
        for p in pose_landmarks
    ], dtype=np.float32)

def split_hands(hand_result):
    left_hand, right_hand = None, None
    for landmarks, categories in zip(hand_result.hand_landmarks, hand_result.handedness):
        handedness = categories[0].category_name.lower() if categories else ""
        if handedness == "left":
            left_hand = landmarks
        elif handedness == "right":
            right_hand = landmarks
    return left_hand, right_hand

def frame_to_features(hand_result, pose_result) -> np.ndarray:
    left_landmarks, right_landmarks = split_hands(hand_result)
    pose_landmarks = pose_result.pose_landmarks[0] if pose_result.pose_landmarks else None
    left = _hand_array(left_landmarks)
    right = _hand_array(right_landmarks)
    pose = _pose_array(pose_landmarks)

    has_left = float(left_landmarks is not None)
    has_right = float(right_landmarks is not None)
    has_pose = float(pose_landmarks is not None)

    if has_pose:
        left_shoulder = pose[11, :3]
        right_shoulder = pose[12, :3]
        center = (left_shoulder + right_shoulder) / 2.0
        scale = max(float(np.linalg.norm(left_shoulder[:2] - right_shoulder[:2])), 1e-4)
        pose[:, :3] = (pose[:, :3] - center) / scale
    else:
        center = np.asarray([0.5, 0.5, 0.0], dtype=np.float32)
        scale = 1.0

    if has_left:
        left = (left - center) / scale
    if has_right:
        right = (right - center) / scale

    features = np.concatenate([
        left.reshape(-1),
        right.reshape(-1),
        pose.reshape(-1),
        np.asarray([has_left, has_right, has_pose], dtype=np.float32),
    ])
    assert features.shape == (FEATURE_DIM,)
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def temporal_resample(sequence: np.ndarray, target_length: int = SEQ_LEN) -> np.ndarray:
    if len(sequence) == 0:
        raise ValueError("Cannot resample an empty sequence.")
    if len(sequence) == 1:
        return np.repeat(sequence, target_length, axis=0).astype(np.float32)
    old_positions = np.linspace(0.0, 1.0, len(sequence))
    new_positions = np.linspace(0.0, 1.0, target_length)
    result = np.stack([
        np.interp(new_positions, old_positions, sequence[:, feature_index])
        for feature_index in range(sequence.shape[1])
    ], axis=1)
    return result.astype(np.float32)

def extract_video_sequence(video_path: Path, landmarkers) -> np.ndarray:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError("OpenCV could not open the video")

    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames > 0:
        selected = set(np.linspace(0, total_frames - 1, min(total_frames, MAX_SAMPLE_FRAMES), dtype=int))
    else:
        selected = None

    frame_features = []
    frame_index = 0
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        should_process = selected is None or frame_index in selected
        if should_process:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            hand_result, pose_result = landmarkers.process(rgb)
            frame_features.append(frame_to_features(hand_result, pose_result))
            if selected is None and len(frame_features) >= MAX_SAMPLE_FRAMES:
                break
        frame_index += 1
    capture.release()

    if not frame_features:
        raise RuntimeError("No readable frames")
    return temporal_resample(np.asarray(frame_features, dtype=np.float32))

def cache_path_for(video_path: Path) -> Path:
    stat = video_path.stat()
    identity = f"{video_path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}|{CACHE_VERSION}|{SEQ_LEN}|{MAX_SAMPLE_FRAMES}"
    key = hashlib.sha1(identity.encode("utf-8")).hexdigest()
    return CACHE_DIR / f"{key}.npz"

def load_or_extract(video_path: Path, landmarkers) -> np.ndarray:
    cache_path = cache_path_for(video_path)
    if cache_path.exists():
        with np.load(cache_path) as cached:
            sequence = cached["sequence"]
        if sequence.shape == (SEQ_LEN, FEATURE_DIM):
            return sequence.astype(np.float32)
    sequence = extract_video_sequence(video_path, landmarkers)
    np.savez_compressed(cache_path, sequence=sequence)
    return sequence

### Landmark preview

Check one middle frame before processing the full dataset. You should see the upper-body pose and visible hand skeletons. If landmarks are often absent, improve lighting/framing or lower the detection confidence slightly.

In [ ]:
def show_landmark_preview(video_path: Path):
    capture = cv2.VideoCapture(str(video_path))
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames > 1:
        capture.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
    ok, frame = capture.read()
    capture.release()
    if not ok:
        raise RuntimeError(f"Could not read preview frame from {video_path}")

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
    with create_hand_landmarker(vision.RunningMode.IMAGE) as hand_detector, \
         create_pose_landmarker(vision.RunningMode.IMAGE) as pose_detector:
        hand_result = hand_detector.detect(image)
        pose_result = pose_detector.detect(image)

    def draw_set(canvas, landmarks, connections, color):
        if not landmarks:
            return
        height, width = canvas.shape[:2]
        for connection in connections:
            start = landmarks[connection.start]
            end = landmarks[connection.end]
            cv2.line(canvas, (int(start.x * width), int(start.y * height)),
                     (int(end.x * width), int(end.y * height)), color, 2)
        for point in landmarks:
            cv2.circle(canvas, (int(point.x * width), int(point.y * height)), 3, color, -1)

    annotated = rgb.copy()
    pose_landmarks = pose_result.pose_landmarks[0] if pose_result.pose_landmarks else None
    draw_set(annotated, pose_landmarks, vision.PoseLandmarksConnections.POSE_LANDMARKS, (0, 220, 0))
    for hand_landmarks in hand_result.hand_landmarks:
        draw_set(annotated, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS, (255, 80, 40))
    plt.figure(figsize=(8, 6))
    plt.imshow(annotated)
    plt.title(f"Landmark preview: {video_path.parent.parent.name} / {video_path.name}")
    plt.axis("off")
    plt.show()

show_landmark_preview(videos_df.iloc[0]["path"])

In [ ]:
sequences, targets, kept_paths, failures = [], [], [], []

with VideoLandmarkers() as landmarkers:
    for row_number, row in enumerate(videos_df.itertuples(index=False), start=1):
        try:
            sequence = load_or_extract(row.path, landmarkers)
            sequences.append(sequence)
            targets.append(row.label_index)
            kept_paths.append(row.path)
        except Exception as exc:
            failures.append({"path": str(row.path), "error": str(exc)})
        if row_number % 10 == 0 or row_number == len(videos_df):
            print(f"Processed {row_number}/{len(videos_df)} videos", end="\r")

X = np.asarray(sequences, dtype=np.float32)
y = np.asarray(targets, dtype=np.int32)
kept_paths = np.asarray(kept_paths, dtype=object)

print(f"\nX shape: {X.shape}; y shape: {y.shape}")
print(f"Failed videos: {len(failures)}")
if failures:
    display(pd.DataFrame(failures))

if len(X) == 0:
    raise RuntimeError("No video produced a usable landmark sequence.")

quality_df = pd.DataFrame({
    "video": [Path(p).name for p in kept_paths],
    "label": [index_to_label[int(i)] for i in y],
    "left_hand_presence": X[:, :, -3].mean(axis=1),
    "right_hand_presence": X[:, :, -2].mean(axis=1),
    "pose_presence": X[:, :, -1].mean(axis=1),
})
display(quality_df.groupby("label")[["left_hand_presence", "right_hand_presence", "pose_presence"]].mean().round(3))
display(quality_df.sort_values(["pose_presence", "left_hand_presence", "right_hand_presence"]).head(10))

## 3. Stratified train/validation/test split

The split happens at the video level, so frames from one recording can never appear in multiple splits. If multiple clips were cut from one long recording, group them by recording/session before splitting to avoid leakage.

In [ ]:
X_train, X_temp, y_train, y_temp, paths_train, paths_temp = train_test_split(
    X, y, kept_paths, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)
X_val, X_test, y_val, y_test, paths_val, paths_test = train_test_split(
    X_temp, y_temp, paths_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

def split_summary(name, values):
    counts = np.bincount(values, minlength=len(labels))
    return pd.Series(counts, index=labels, name=name)

display(pd.concat([
    split_summary("train", y_train),
    split_summary("validation", y_val),
    split_summary("test", y_test),
], axis=1))
print("Split sizes:", len(X_train), len(X_val), len(X_test))

## 4. Build and train the temporal model

Normalization statistics are calculated only from the training split and later exported for identical browser preprocessing. Class weights reduce bias when one word has more recordings than another. Early stopping restores the best validation checkpoint.

In [ ]:
flat_train = X_train.reshape(-1, FEATURE_DIM)
feature_mean = flat_train.mean(axis=0).astype(np.float32)
feature_std = flat_train.std(axis=0).astype(np.float32)
feature_std = np.where(feature_std < 1e-6, 1.0, feature_std).astype(np.float32)

def standardize_sequences(values):
    return ((values - feature_mean) / feature_std).astype(np.float32)

X_train_scaled = standardize_sequences(X_train)
X_val_scaled = standardize_sequences(X_val)
X_test_scaled = standardize_sequences(X_test)

inputs = keras.Input(shape=(SEQ_LEN, FEATURE_DIM), name="landmark_sequence")
x = layers.Conv1D(96, kernel_size=3, padding="same", activation="relu")(inputs)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.Conv1D(160, kernel_size=3, padding="same", activation="relu")(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(96, activation="relu")(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(len(labels), activation="softmax", name="word_probabilities")(x)

model = keras.Model(inputs, outputs, name="word_sign_temporal_cnn")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

class_values = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=class_values, y=y_train)
class_weight = {int(class_id): float(weight) for class_id, weight in zip(class_values, weights)}
class_weight

In [ ]:
BEST_MODEL_PATH = OUTPUT_DIR / "best_word_sign_model.keras"
callbacks = [
    keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor="val_accuracy", mode="max", save_best_only=True, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=12, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
    ),
]

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=8,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
for axis in axes:
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Evaluate on unseen videos

Use the held-out test set only for the final estimate. With a small dataset, per-class recall and the confusion matrix are more informative than accuracy alone.

In [ ]:
model = keras.models.load_model(BEST_MODEL_PATH)
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
probabilities = model.predict(X_test_scaled, verbose=0)
predictions = probabilities.argmax(axis=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")
print()
print(classification_report(
    y_test, predictions, labels=np.arange(len(labels)), target_names=labels, zero_division=0
))

matrix = confusion_matrix(y_test, predictions, labels=np.arange(len(labels)))
plt.figure(figsize=(7, 6))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted word")
plt.ylabel("True word")
plt.title("Test confusion matrix")
plt.tight_layout()
plt.show()

results_df = pd.DataFrame({
    "video": [Path(p).name for p in paths_test],
    "actual": [index_to_label[int(i)] for i in y_test],
    "predicted": [index_to_label[int(i)] for i in predictions],
    "confidence": probabilities.max(axis=1),
})
display(results_df.sort_values("confidence"))

## 6. Save the deployable artifacts

The `.keras` file contains the feature normalizer and neural network. The JSON files preserve the label order and preprocessing settings needed by an app.

In [ ]:
FINAL_MODEL_PATH = OUTPUT_DIR / "word_sign_model.keras"
LABELS_PATH = OUTPUT_DIR / "labels.json"
CONFIG_PATH = OUTPUT_DIR / "feature_config.json"

model.save(FINAL_MODEL_PATH)
with LABELS_PATH.open("w", encoding="utf-8") as file:
    json.dump(labels, file, indent=2, ensure_ascii=False)
with CONFIG_PATH.open("w", encoding="utf-8") as file:
    json.dump({
        "sequence_length": SEQ_LEN,
        "feature_dimension": FEATURE_DIM,
        "max_sample_frames": MAX_SAMPLE_FRAMES,
        "cache_version": CACHE_VERSION,
        "feature_order": ["left_hand_21x3", "right_hand_21x3", "pose_33x4", "presence_flags_3"],
        "normalization": "shoulder-center/width followed by per-feature z-score",
        "feature_mean": feature_mean.tolist(),
        "feature_std": feature_std.tolist(),
    }, file, indent=2)

print("Saved:")
print(" -", FINAL_MODEL_PATH)
print(" -", LABELS_PATH)
print(" -", CONFIG_PATH)

## 7. Export for JavaScript / Web (TensorFlow.js)

This writes a TensorFlow.js weight bundle with `model.json`, a binary weight shard, `labels.json`, and `feature_config.json`—without requiring the Python `tensorflowjs` converter (which is currently incompatible with Python 3.13). In Colab the files go to `MyDrive/sign_language_model/web_export/models/word-sign/`; copy that folder into the web project's `public/models/word-sign/` directory. A ready-to-download ZIP is also created in Drive.

In [ ]:
TFJS_DIR = WEB_EXPORT_DIR
TFJS_DIR.mkdir(parents=True, exist_ok=True)

# Remove only old artifacts produced by this export, preventing stale shards.
for artifact in TFJS_DIR.glob("group*-shard*.bin"):
    artifact.unlink()
(TFJS_DIR / "model.json").unlink(missing_ok=True)

# TF.js Layers orders all trainable variables before non-trainable BatchNorm statistics.
keras_weight_variables = list(model.trainable_weights) + list(model.non_trainable_weights)
weight_names = [f"weight_{index:03d}" for index in range(len(keras_weight_variables))]
weight_arrays = [np.asarray(weight.numpy(), dtype="<f4") for weight in keras_weight_variables]
weight_specs = [
    {"name": name, "shape": list(weight.shape), "dtype": "float32"}
    for name, weight in zip(weight_names, weight_arrays)
]
with (TFJS_DIR / "group1-shard1of1.bin").open("wb") as file:
    for weight in weight_arrays:
        file.write(weight.tobytes(order="C"))

browser_manifest = {
    "format": "handspeak-temporal-cnn-v1",
    "generatedBy": f"TensorFlow {tf.__version__}",
    "modelTopology": {
        "inputShape": [SEQ_LEN, FEATURE_DIM],
        "classCount": len(labels),
        "convFilters": [96, 128, 160],
        "denseUnits": 96,
    },
    "weightOrder": weight_names,
    "weightsManifest": [{
        "paths": ["group1-shard1of1.bin"],
        "weights": weight_specs,
    }],
}
with (TFJS_DIR / "model.json").open("w", encoding="utf-8") as file:
    json.dump(browser_manifest, file, indent=2)

shutil.copy2(LABELS_PATH, TFJS_DIR / "labels.json")
shutil.copy2(CONFIG_PATH, TFJS_DIR / "feature_config.json")
TFJS_ZIP_BASE = PROJECT_DIR / "word-sign-tfjs"
TFJS_ZIP_PATH = Path(shutil.make_archive(str(TFJS_ZIP_BASE), "zip", root_dir=TFJS_DIR))

artifacts = sorted(path.name for path in TFJS_DIR.iterdir() if path.is_file())
assert "model.json" in artifacts and any(name.endswith(".bin") for name in artifacts)
print("TensorFlow.js export complete:", TFJS_DIR)
print("Browser URL: /models/word-sign/model.json")
print("Downloadable ZIP:", TFJS_ZIP_PATH)
print("Files:", artifacts)

### Browser usage

The project includes `src/ai/wordSignModel.js`, which applies the same feature construction, temporal resampling, and normalization as this notebook. Pass it the two hand landmark arrays and pose array returned by MediaPipe Tasks Vision:

```javascript
import {
  loadWordSignModel,
  buildWordFrameFeatures,
  predictWordSequence
} from './ai/wordSignModel.js';

await loadWordSignModel();
const capturedFrames = [];

// Call while recording one complete sign:
capturedFrames.push(buildWordFrameFeatures({ leftHand, rightHand, pose }));

// Call when the recording window is complete:
const result = await predictWordSequence(capturedFrames, 0.60);
console.log(result.label, result.confidence, result.ranking);
```

Use the included `loadWordSignModel()` helper. It reconstructs the temporal CNN in TensorFlow.js and loads the exported binary weights with `tf.io.loadWeights()`.

## 8. Predict one video in Python

This uses the exact same MediaPipe and temporal preprocessing as training. A low confidence can be treated as `Unknown` in an application.

In [ ]:
def predict_video(video_path, confidence_threshold=0.60):
    video_path = Path(video_path)
    inference_model = keras.models.load_model(FINAL_MODEL_PATH)
    with LABELS_PATH.open(encoding="utf-8") as file:
        inference_labels = json.load(file)
    with CONFIG_PATH.open(encoding="utf-8") as file:
        inference_config = json.load(file)

    with VideoLandmarkers() as landmarkers:
        sequence = extract_video_sequence(video_path, landmarkers)

    mean = np.asarray(inference_config["feature_mean"], dtype=np.float32)
    std = np.asarray(inference_config["feature_std"], dtype=np.float32)
    sequence = ((sequence - mean) / std).astype(np.float32)
    scores = inference_model.predict(sequence[None, ...], verbose=0)[0]
    best_index = int(np.argmax(scores))
    confidence = float(scores[best_index])
    predicted_word = inference_labels[best_index] if confidence >= confidence_threshold else "Unknown"
    ranking = sorted(
        ((inference_labels[i], float(score)) for i, score in enumerate(scores)),
        key=lambda item: item[1], reverse=True
    )
    return {"word": predicted_word, "confidence": confidence, "ranking": ranking}

example_video = paths_test[0]
result = predict_video(example_video)
print("Video:", example_video)
print("Prediction:", result["word"], f"({result['confidence']:.1%})")
print("All scores:", result["ranking"])

## Practical notes

- Keep the full upper body and both hands visible. Avoid starting or ending recordings mid-sign.
- Collect examples from several people. If you do, split by signer rather than randomly by video for an honest evaluation.
- Include an `Unknown`/`Other` class with non-target movements before using the model live; otherwise softmax must choose one known word.
- Similar phrases such as **Good Morning** are sequences of words. This notebook classifies one trimmed word per video; continuous sentence recognition needs segmentation or a sequence-to-sequence/CTC model.
- If landmark quality is low, improve framing and lighting before increasing model size.